# LIME Explainability Analysis — MGT Detection
**COS760 Group 57 | Feature importance comparison across languages**

---

This notebook runs LIME on the fine-tuned AfroXLMR models to answer RQ2:
What linguistic features drive detection in isiZulu vs English, and how do they differ in cross-lingual transfer?

### Before running:
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. You need the saved model checkpoints from training (best_model folders)
3. Upload test CSV files when prompted

## Cell 1 — Install packages

In [ ]:
!pip install transformers datasets scikit-learn lime matplotlib numpy pandas scipy accelerate -q
print('Packages installed')

## Cell 2 — Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('No GPU — LIME will be slow but still works on CPU')

## Cell 3 — Upload files
Upload your test CSVs (`zul_test.csv`, `eng_test.csv`) and model checkpoint folders.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
os.makedirs('/content/data', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'/content/data/{fname}')
print('Files uploaded:', os.listdir('/content/data'))

## Cell 3b — Upload model checkpoint
Upload the `best_model` folder contents (config.json, model.safetensors, tokenizer files) from your AfroXLMR isiZulu training run.

In [ ]:
os.makedirs('/content/model_zul', exist_ok=True)
print('Upload the best_model files from your isiZulu AfroXLMR run:')
model_files = files.upload()
for fname in model_files:
    os.rename(fname, f'/content/model_zul/{fname}')
print('Model files:', os.listdir('/content/model_zul'))

os.makedirs('/content/model_cross', exist_ok=True)
print('\nNow upload the best_model files from your cross-lingual (English-trained) run:')
model_files2 = files.upload()
for fname in model_files2:
    os.rename(fname, f'/content/model_cross/{fname}')
print('Cross model files:', os.listdir('/content/model_cross'))

## Cell 4 — Load models and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
import scipy.special

# Load isiZulu-trained model
tokenizer = AutoTokenizer.from_pretrained('/content/model_zul')
model_zul = AutoModelForSequenceClassification.from_pretrained('/content/model_zul')
model_zul.eval()
model_zul.cuda()

# Load cross-lingual model (English-trained)
model_cross = AutoModelForSequenceClassification.from_pretrained('/content/model_cross')
model_cross.eval()
model_cross.cuda()

print('Both models loaded')

## Cell 5 — Define prediction functions for LIME

In [ ]:
def make_predictor(model):
    """Create a prediction function compatible with LIME."""
    def predict_proba(texts):
        all_probs = []
        batch_size = 16
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch, padding=True, truncation=True,
                max_length=256, return_tensors='pt'
            ).to('cuda')
            with torch.no_grad():
                logits = model(**inputs).logits.cpu().numpy()
            probs = scipy.special.softmax(logits, axis=-1)
            all_probs.append(probs)
        return np.vstack(all_probs)
    return predict_proba

predict_zul = make_predictor(model_zul)
predict_cross = make_predictor(model_cross)
print('Prediction functions ready')

## Cell 6 — Load test data and sample instances

In [ ]:
zul_test = pd.read_csv('/content/data/zul_test.csv')
eng_test = pd.read_csv('/content/data/eng_test.csv')

# Sample instances for LIME (balanced: human + machine)
N_SAMPLES = 20  # 20 per class per language = 80 total explanations

np.random.seed(42)

zul_human = zul_test[zul_test['label'] == 0].sample(N_SAMPLES, random_state=42)
zul_machine = zul_test[zul_test['label'] == 1].sample(N_SAMPLES, random_state=42)
eng_human = eng_test[eng_test['label'] == 0].sample(N_SAMPLES, random_state=42)
eng_machine = eng_test[eng_test['label'] == 1].sample(N_SAMPLES, random_state=42)

print(f'Sampled {N_SAMPLES} human + {N_SAMPLES} machine per language')
print(f'Total LIME explanations to generate: {N_SAMPLES * 4}')

## Cell 7 — Run LIME explanations
This takes ~15-30 minutes depending on sample size. Generates explanations for both models on isiZulu text.

In [ ]:
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=['Human', 'Machine'], split_expression=r'\s+')

def get_explanations(texts, predictor, num_features=10, num_samples=500):
    """Generate LIME explanations for a list of texts."""
    explanations = []
    for i, text in enumerate(texts):
        if i % 5 == 0:
            print(f'  Explaining {i+1}/{len(texts)}...')
        exp = explainer.explain_instance(
            text, predictor,
            num_features=num_features,
            num_samples=num_samples
        )
        explanations.append(exp)
    return explanations

print('=== isiZulu model on isiZulu human text ===')
exp_zul_human = get_explanations(zul_human['text'].tolist(), predict_zul)

print('\n=== isiZulu model on isiZulu machine text ===')
exp_zul_machine = get_explanations(zul_machine['text'].tolist(), predict_zul)

print('\n=== Cross-lingual model on isiZulu human text ===')
exp_cross_human = get_explanations(zul_human['text'].tolist(), predict_cross)

print('\n=== Cross-lingual model on isiZulu machine text ===')
exp_cross_machine = get_explanations(zul_machine['text'].tolist(), predict_cross)

print('\nAll explanations generated!')

## Cell 8 — Aggregate feature importance

In [ ]:
from collections import defaultdict

def aggregate_features(explanations, top_k=20):
    """Aggregate feature weights across multiple LIME explanations."""
    feature_weights = defaultdict(list)
    for exp in explanations:
        for word, weight in exp.as_list():
            feature_weights[word].append(weight)

    # Average weight and frequency
    summary = []
    for word, weights in feature_weights.items():
        summary.append({
            'feature': word,
            'mean_weight': np.mean(weights),
            'abs_mean_weight': np.abs(np.mean(weights)),
            'frequency': len(weights),
            'std': np.std(weights)
        })

    df = pd.DataFrame(summary).sort_values('abs_mean_weight', ascending=False)
    return df.head(top_k)

# Aggregate for each condition
feat_zul_human = aggregate_features(exp_zul_human)
feat_zul_machine = aggregate_features(exp_zul_machine)
feat_cross_human = aggregate_features(exp_cross_human)
feat_cross_machine = aggregate_features(exp_cross_machine)

print('Top features — isiZulu model on MACHINE text:')
print(feat_zul_machine[['feature', 'mean_weight', 'frequency']].to_string(index=False))
print('\nTop features — Cross-lingual model on MACHINE text:')
print(feat_cross_machine[['feature', 'mean_weight', 'frequency']].to_string(index=False))

## Cell 9 — Visualise: Top features comparison

In [ ]:
import matplotlib.pyplot as plt

def plot_top_features(df, title, ax, color='steelblue'):
    """Plot horizontal bar chart of top LIME features."""
    top = df.head(15).sort_values('mean_weight')
    colors = ['#d32f2f' if w > 0 else '#1976d2' for w in top['mean_weight']]
    ax.barh(top['feature'], top['mean_weight'], color=colors)
    ax.set_xlabel('Mean LIME Weight')
    ax.set_title(title, fontsize=11)
    ax.axvline(x=0, color='black', linewidth=0.5)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

plot_top_features(feat_zul_human, 'isiZulu Model — Human Text Features', axes[0, 0])
plot_top_features(feat_zul_machine, 'isiZulu Model — Machine Text Features', axes[0, 1])
plot_top_features(feat_cross_human, 'Cross-lingual Model — Human Text Features', axes[1, 0])
plot_top_features(feat_cross_machine, 'Cross-lingual Model — Machine Text Features', axes[1, 1])

plt.suptitle('LIME Feature Importance: isiZulu vs Cross-lingual Detection', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/content/lime_features_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lime_features_comparison.png')

## Cell 10 — Individual example explanations

In [ ]:
# Show a single example explanation (isiZulu machine text)
print('Example: isiZulu machine-generated text')
print(f'Text: {zul_machine.iloc[0]["text"][:200]}...')
print()

fig = exp_zul_machine[0].as_pyplot_figure()
plt.title('LIME — isiZulu Model on Machine Text (Example 1)')
plt.tight_layout()
plt.savefig('/content/lime_example_zul_machine.png', dpi=150, bbox_inches='tight')
plt.show()

# Cross-lingual on same text
print('\nSame text — Cross-lingual model:')
fig = exp_cross_machine[0].as_pyplot_figure()
plt.title('LIME — Cross-lingual Model on Machine Text (Example 1)')
plt.tight_layout()
plt.savefig('/content/lime_example_cross_machine.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 — Feature overlap analysis
How much do the two models agree on which features matter?

In [ ]:
# Compare top features between isiZulu model and cross-lingual model
top_zul = set(feat_zul_machine.head(20)['feature'].tolist())
top_cross = set(feat_cross_machine.head(20)['feature'].tolist())

overlap = top_zul & top_cross
only_zul = top_zul - top_cross
only_cross = top_cross - top_zul

print(f'Top-20 feature overlap: {len(overlap)}/20 ({len(overlap)/20*100:.0f}%)')
print(f'\nShared features: {sorted(overlap)}')
print(f'\nOnly in isiZulu model: {sorted(only_zul)}')
print(f'\nOnly in cross-lingual model: {sorted(only_cross)}')

# Jaccard similarity
jaccard = len(overlap) / len(top_zul | top_cross)
print(f'\nJaccard similarity: {jaccard:.3f}')

## Cell 12 — Save all results and download

In [ ]:
from google.colab import files as colab_files
import zipfile

# Save feature tables
feat_zul_machine.to_csv('/content/lime_features_zul_model_machine.csv', index=False)
feat_zul_human.to_csv('/content/lime_features_zul_model_human.csv', index=False)
feat_cross_machine.to_csv('/content/lime_features_cross_model_machine.csv', index=False)
feat_cross_human.to_csv('/content/lime_features_cross_model_human.csv', index=False)

# Zip everything
with zipfile.ZipFile('/content/lime_results.zip', 'w') as zf:
    zf.write('/content/lime_features_zul_model_machine.csv', 'lime_features_zul_model_machine.csv')
    zf.write('/content/lime_features_zul_model_human.csv', 'lime_features_zul_model_human.csv')
    zf.write('/content/lime_features_cross_model_machine.csv', 'lime_features_cross_model_machine.csv')
    zf.write('/content/lime_features_cross_model_human.csv', 'lime_features_cross_model_human.csv')
    zf.write('/content/lime_features_comparison.png', 'lime_features_comparison.png')
    zf.write('/content/lime_example_zul_machine.png', 'lime_example_zul_machine.png')
    zf.write('/content/lime_example_cross_machine.png', 'lime_example_cross_machine.png')

colab_files.download('/content/lime_results.zip')
print('Downloaded: lime_results.zip')